<a href="https://colab.research.google.com/github/brasilemnumeros/dados/blob/main/indicadores/igp-m/IGP-M.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# IGP-M — acumulado em 12 meses

Este notebook é o motor de coleta e publicação da série `igpm-acum-12m`.

- **Fonte:** FGV (IGP-M), via API de Dados Abertos do Banco Central (SGS), série **28655** —
  variação mensal do IGP-M calculada sobre os números-índices da FGV (3 casas decimais). É a
  mesma série que o BCB usa na Calculadora do Cidadão para correção de valores pelo IGP-M
  desde 12/04/2021.
- **Endpoint:** `https://api.bcb.gov.br/dados/serie/bcdata.sgs.28655/dados?formato=json`
- **Definição:** variação percentual do IGP-M acumulada nos últimos 12 meses (taxa efetiva do
  período), unidade `percentual` (`%`, 2 casas decimais), `medida: acumulado-12m`,
  granularidade `mensal`.
- **Fórmula:** composição geométrica das 12 variações mensais —
  `(∏(1 + i/100) − 1) × 100` —, nunca a soma aritmética das taxas.
- **Período publicado:** começa em `1995-01-01` e termina no último mês fechado.
- **Arquivo gerado:** `indicadores/igp-m/igpm-acum-12m.json`, com `versao: 1` e
  `id: igpm-acum-12m`.

## Decisão metodológica

O IGP-M **não tem série pronta de acumulado em 12 meses** no SGS: o catálogo de séries do IGP-M
(`https://www4.bcb.gov.br/pec/series/port/metadados/mg40Bp.htm`) lista apenas 0189 (variação
mensal), 7448/7449 (1º e 2º decêndios) e 7450–7458 (componentes IPA-M, IPC-M, INCC-M) —
nenhuma acumulada em 12 meses. Por isso o acumulado é composto aqui, a partir da SGS **28655**.

A 28655 é a variação mensal com precisão estendida, derivada dos números-índices da FGV; ela
reproduz os valores que a FGV publica para 12 meses melhor do que a 0189 (2 casas): compor a
28655 acerta os 14 meses da tabela de 12 meses divulgada pela FGV (nov/2024 a dez/2025), contra
8 usando a 0189. A 0189 é
buscada apenas para conferência (as duas coincidem até a 2ª casa em toda a história), nunca
para gerar a saída. Ver `metodologia.md`, seção `igpm-acum-12m`.

## Execução sem sobrescrever o publicado

A variável de ambiente `IGPM_SAIDA` redireciona o arquivo de saída. Para rodar em ensaio:

```
IGPM_SAIDA=/caminho/temporario jupyter nbconvert --execute --to notebook IGP-M.ipynb
```


## Coleta da série mensal oficial

A leitura é feita desde o início da série, sem atualização incremental e sem ler o JSON já
publicado para decidir o período.


In [1]:
from datetime import date
from decimal import Decimal, ROUND_HALF_UP
from pathlib import Path
import json
import math
import time
import os

import requests

RAIZ = Path(os.environ.get("DADOS_RAIZ", Path.cwd())).resolve()

def relativo(caminho):
    """Caminho exibido no notebook: relativo à raiz, nunca o absoluto da máquina."""
    try:
        return caminho.relative_to(RAIZ)
    except ValueError:
        return caminho.name
PASTA = RAIZ / "indicadores" / "igp-m" if (RAIZ / "indicadores" / "igp-m").is_dir() else RAIZ
DESTINO = Path(os.environ.get("IGPM_SAIDA", PASTA))
ARQUIVO = DESTINO / "igpm-acum-12m.json"

SERIE = 28655
SERIE_CONFERENCIA = 189
URL = "https://api.bcb.gov.br/dados/serie/bcdata.sgs.{serie}/dados"
URL_FONTE = URL.format(serie=SERIE) + "?formato=json"
INICIO = date(1995, 1, 1)
HOJE = date.today()


def baixar(serie, tentativas=4):
    """Baixa uma série SGS e devolve [(date, float)] em ordem ascendente.

    O SGS devolve 502 esporadicamente; a leitura é repetida com espera
    crescente antes de desistir.
    """
    for tentativa in range(tentativas):
        try:
            resposta = requests.get(
                URL.format(serie=serie), params={"formato": "json"}, timeout=60
            )
            resposta.raise_for_status()
            bruto = resposta.json()
            break
        except (requests.RequestException, ValueError):
            if tentativa == tentativas - 1:
                raise
            time.sleep(2 ** tentativa)
    if not bruto:
        raise ValueError(f"a série SGS {serie} não retornou dados")
    leituras = []
    for item in bruto:
        dia, mes, ano = (int(parte) for parte in item["data"].split("/"))
        leituras.append((date(ano, mes, dia), float(item["valor"])))
    leituras.sort()
    return leituras


def indice_mensal(dia):
    """Mês absoluto: serve para exigir um ponto por mês, sem lacuna."""
    return dia.year * 12 + (dia.month - 1)


mensal = baixar(SERIE)
mensal_por_data = dict(mensal)
conferencia = dict(baixar(SERIE_CONFERENCIA))

if len(mensal) < 12:
    raise ValueError(f"a SGS {SERIE} tem menos de 12 meses: {len(mensal)}")

if len({dia for dia, _ in mensal}) != len(mensal):
    raise ValueError(f"a SGS {SERIE} retornou meses repetidos")

for (anterior, _), (seguinte, _) in zip(mensal, mensal[1:]):
    if indice_mensal(seguinte) - indice_mensal(anterior) != 1:
        raise ValueError(f"lacuna na SGS {SERIE}: {anterior} -> {seguinte}")

print(f"Série SGS {SERIE}: {len(mensal)} leituras")
print(f"Período bruto: {mensal[0][0]:%Y-%m-%d} a {mensal[-1][0]:%Y-%m-%d}")
print(f"Arquivo de saída: {relativo(ARQUIVO)}")

divergencias = [
    (dia, valor, conferencia[dia])
    for dia, valor in mensal
    if dia in conferencia and abs(conferencia[dia] - valor) > 0.01
]
print(f"\nSGS {SERIE} x SGS {SERIE_CONFERENCIA}: {len(divergencias)} divergências acima de 0,01 p.p.")
print("\nPrimeiras 3 leituras mensais:")
for dia, valor in mensal[:3]:
    print(f"  {dia:%Y-%m-%d}  {valor:12.6f}%")
print("Últimas 3 leituras mensais:")
for dia, valor in mensal[-3:]:
    print(f"  {dia:%Y-%m-%d}  {valor:12.6f}%")


Série SGS 28655: 449 leituras
Período bruto: 1989-06-01 a 2026-10-01
Arquivo de saída: indicadores/igp-m/igpm-acum-12m.json

SGS 28655 x SGS 189: 0 divergências acima de 0,01 p.p.

Primeiras 3 leituras mensais:
  1989-06-01     19.680000%
  1989-07-01     35.905113%
  1989-08-01     36.916141%
Últimas 3 leituras mensais:
  2026-08-01     -0.224865%
  2026-09-01      1.570059%
  2026-10-01      1.473187%


## Composição do acumulado em 12 meses

Cada ponto é a taxa efetiva dos 12 meses terminados naquele mês: o produto dos fatores
`(1 + i/100)` das 12 variações mensais, menos 1, em porcentagem. Os 11 primeiros meses da
série não têm janela completa e não são publicados. O mês corrente é descartado porque ainda
está aberto.


In [2]:
def acumulado_12m(leituras):
    """Composição geométrica: (∏(1 + i/100) − 1) × 100, nunca soma."""
    serie = []
    for posicao in range(11, len(leituras)):
        janela = leituras[posicao - 11 : posicao + 1]
        fator = math.prod(1 + taxa / 100 for _, taxa in janela)
        serie.append((janela[-1][0], (fator - 1) * 100))
    return serie


def arredondar(valor):
    """Arredonda para 2 casas com meio para cima (convenção comercial)."""
    return float(Decimal(repr(valor)).quantize(Decimal("0.01"), rounding=ROUND_HALF_UP))


primeiro_dia_mes_corrente = HOJE.replace(day=1)
bruto_12m = acumulado_12m(mensal)
fechados = [
    (dia, valor)
    for dia, valor in bruto_12m
    if INICIO <= dia < primeiro_dia_mes_corrente
]

if not fechados:
    raise ValueError("nenhum mês fechado no período publicado")

print(f"Janelas completas na fonte: {len(bruto_12m)} ({bruto_12m[0][0]:%Y-%m} a {bruto_12m[-1][0]:%Y-%m})")
print(f"Mesmo assim, a série publicada começa em {INICIO:%Y-%m}:")
print("o recorte alinha o IGP-M ao ipca-acum-12m, que também começa em 1995-01,")
print("e deixa de fora a cauda de hiperinflação (1990-1994, até milhares por cento ao ano).")
print()
print(f"Mês corrente excluído: {HOJE:%Y-%m}")
print(f"Meses fechados publicados: {len(fechados)}")
print(f"Primeiro: {fechados[0][0]:%Y-%m-%d} ({fechados[0][1]:.4f}%)")
print(f"Último:   {fechados[-1][0]:%Y-%m-%d} ({fechados[-1][1]:.4f}%)")


Janelas completas na fonte: 438 (1990-05 a 2026-10)
Mesmo assim, a série publicada começa em 1995-01:
o recorte alinha o IGP-M ao ipca-acum-12m, que também começa em 1995-01,
e deixa de fora a cauda de hiperinflação (1990-1994, até milhares por cento ao ano).

Mês corrente excluído: 2026-11
Meses fechados publicados: 382
Primeiro: 1995-01-01 (603.7419%)
Último:   2026-10-01 (5.2369%)


## Escrita do JSON do contrato

O arquivo é gravado inteiro (nada de merge com o que já está publicado), com `versao`, `id`,
`unidade`, `eixo`, `medida`, `granularidade`, `fonte`, `urlFonte` e `dados` no formato
`{"data": "YYYY-MM-DD", "valor": ...}`.


In [3]:
pontos = [
    {"data": dia.strftime("%Y-%m-%d"), "valor": arredondar(valor)}
    for dia, valor in fechados
]

serie = {
    "versao": 1,
    "id": "igpm-acum-12m",
    "unidade": {"tipo": "percentual", "simbolo": "%", "casas": 2},
    "eixo": "percentual",
    "medida": "acumulado-12m",
    "granularidade": "mensal",
    "fonte": "FGV (SGS 28655); acumulado 12m calculado pelo site",
    "urlFonte": URL_FONTE,
    "dados": pontos,
}

DESTINO.mkdir(parents=True, exist_ok=True)
with ARQUIVO.open("w", encoding="utf-8") as arquivo:
    json.dump(serie, arquivo, ensure_ascii=False, indent=2)
    arquivo.write("\n")

print(f"{len(pontos)} pontos escritos em {ARQUIVO}")
print(f"Primeiro ponto: {pontos[0]}")
print(f"Último ponto:   {pontos[-1]}")


382 pontos escritos em indicadores/igp-m/igpm-acum-12m.json
Primeiro ponto: {'data': '1995-01-01', 'valor': 603.74}
Último ponto:   {'data': '2026-10-01', 'valor': 5.24}


## Validação inline

A célula final falha se o contrato, a ordem, a unicidade e a continuidade dos meses, a
finitude dos valores, a composição ou a igualdade com a fonte forem violados. Ela também
recalcula o acumulado a partir da fonte e confere os números citados em `metodologia.md`
e na tabela de 12 meses publicada pela FGV.


In [4]:
CHAVES_RAIZ = {
    "versao", "id", "unidade", "eixo", "medida", "granularidade", "fonte", "urlFonte", "dados",
}
assert set(serie) == CHAVES_RAIZ, sorted(set(serie) ^ CHAVES_RAIZ)
assert serie["versao"] == 1, serie["versao"]
assert serie["id"] == "igpm-acum-12m", serie["id"]
assert serie["unidade"] == {"tipo": "percentual", "simbolo": "%", "casas": 2}, serie["unidade"]
assert serie["eixo"] == "percentual", serie["eixo"]
assert serie["medida"] == "acumulado-12m", serie["medida"]
assert serie["granularidade"] == "mensal", serie["granularidade"]
assert serie["fonte"] == "FGV (SGS 28655); acumulado 12m calculado pelo site", serie["fonte"]
assert serie["urlFonte"].startswith("https://"), serie["urlFonte"]

dados = serie["dados"]
assert dados, "série vazia"

datas = [ponto["data"] for ponto in dados]
dias = [date.fromisoformat(texto) for texto in datas]
por_data = {ponto["data"]: ponto["valor"] for ponto in dados}

for ponto in dados:
    assert set(ponto) == {"data", "valor"}, sorted(ponto)
    assert isinstance(ponto["data"], str), ponto
    assert not isinstance(ponto["valor"], bool), ponto
    assert isinstance(ponto["valor"], (int, float)), ponto
    assert math.isfinite(ponto["valor"]), ponto

assert all(a < b for a, b in zip(dias, dias[1:])), "datas fora de ordem ou repetidas"
assert all(dia.day == 1 for dia in dias), "dia diferente de 01"
assert all(
    indice_mensal(b) - indice_mensal(a) == 1 for a, b in zip(dias, dias[1:])
), "mês faltando ou repetido"

assert datas[0] == "1995-01-01", datas[0]
assert dias[-1] < primeiro_dia_mes_corrente, datas[-1]
assert len(dados) == len(fechados), (len(dados), len(fechados))

# 1) Cada valor é a composição dos 12 meses da fonte, arredondada a 2 casas.
esperado = [
    {"data": dia.strftime("%Y-%m-%d"), "valor": arredondar(valor)}
    for dia, valor in fechados
]
assert dados == esperado, "valor publicado difere da composição da fonte"

# 2) A composição nunca é a soma aritmética das 12 taxas (ex.: dez/2002).
posicao = {dia: i for i, (dia, _) in enumerate(mensal)}
alvo = date(2002, 12, 1)
soma = sum(mensal[i][1] for i in range(posicao[alvo] - 11, posicao[alvo] + 1))
composto = por_data["2002-12-01"]
assert abs(composto - soma) > 2, (alvo, composto, soma)

# 3) A fonte tem precisão estendida: confere com a variação mensal publicada (0189).
comuns = sorted(set(conferencia) & set(mensal_por_data))
assert len(comuns) >= len(mensal) - 1, "séries com cobertura diferente"
for dia in comuns:
    assert abs(conferencia[dia] - mensal_por_data[dia]) <= 0.01, (
        dia, mensal_por_data[dia], conferencia[dia],
    )

# 4) Conferência contra a tabela de 12 meses publicada pela FGV (2024-2025).
fgv = {
    "2024-11-01": 6.33,
    "2024-12-01": 6.54,
    "2025-01-01": 6.75,
    "2025-02-01": 8.44,
    "2025-03-01": 8.58,
    "2025-04-01": 8.50,
    "2025-05-01": 7.02,
    "2025-06-01": 4.39,
    "2025-07-01": 2.96,
    "2025-08-01": 3.03,
    "2025-09-01": 2.82,
    "2025-10-01": 0.92,
    "2025-11-01": -0.11,
    "2025-12-01": -1.05,
}
for data_fgv, valor_fgv in fgv.items():
    assert abs(por_data[data_fgv] - valor_fgv) <= 0.01, (data_fgv, por_data[data_fgv], valor_fgv)

# 5) Números citados em metodologia.md (choque de 2002-2003).
assert por_data["2002-12-01"] == 25.31, por_data["2002-12-01"]
assert por_data["2003-02-01"] == 30.60, por_data["2003-02-01"]
assert por_data["2025-06-01"] == 4.39, por_data["2025-06-01"]

with ARQUIVO.open(encoding="utf-8") as arquivo:
    salvo = json.load(arquivo)
assert salvo == serie, "arquivo gravado difere do objeto validado"

print(
    f"ok: {len(dados)} meses únicos e contínuos, {datas[0]} a {datas[-1]}, "
    f"composição de (1 + i/100) da SGS {SERIE}, último valor {dados[-1]['valor']:.2f}%"
)


ok: 382 meses únicos e contínuos, 1995-01-01 a 2026-10-01, composição de (1 + i/100) da SGS 28655, último valor 5.24%
